In [29]:
!pip install sentence-transformers
!pip install faiss-cpu
!pip install transformers accelerate


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


# Assignment Introduction: Understanding Retrieval-Augmented Generation (RAG)

Large Language Models (LLMs) are powerful, but they have fundamental limitations. They can only use the knowledge stored inside their parameters, which means they:

- cannot access new or updated information
- cannot reference external documents
- may hallucinate facts that do not exist
- cannot provide grounded or verifiable sources

To address these issues, modern AI systems use a technique called **Retrieval-Augmented Generation (RAG)**.

---

## What is RAG?

Retrieval-Augmented Generation combines two components:

### 1. Retriever
The retriever searches an external knowledge base and finds the most relevant text chunks for a given query.  
It requires the following steps:

- splitting documents into smaller chunks  
- embedding text into numerical vectors  
- computing similarity between vectors  
- selecting the top-k relevant chunks  

### 2. Generator
The generator is the LLM that uses the retrieved text to produce a more accurate and grounded answer.

By adding retrieved context to the prompt, the LLM is able to answer questions using information that is not stored in its parameters.

---

## Why RAG Matters

RAG significantly improves the behavior of LLMs:

- reduces hallucinations  
- provides factual grounding and citations  
- enables domain-specific knowledge (medical, legal, academic, etc.)  
- allows the model to use new information without retraining  
- improves accuracy on specialized tasks  

---

## Why This Assignment?

In this assignment, you will **implement a RAG pipeline from scratch**, without using FAISS, LangChain, or other retrieval libraries. You can use numpy.
You will manually build each major component:

1. manual chunking  
2. sentence-transformer embeddings  
3. manual cosine similarity  
4. manual retrieval  
5. prompt construction  
6. generation with a small HuggingFace model  
7. comparison between RAG and non-RAG answers  

This step-by-step approach helps you see exactly how retrieval works and why it improves model outputs.

---

## What You Will Submit

You will submit:

- a completed Jupyter/Colab notebook  
- answers to conceptual questions  
- a comparison between:
  - LLM without RAG  
  - LLM with RAG  
- one small extension experiment (e.g., add your own documents)

---

## 📌 Section 1 — Create Small Knowledge Base (Manual Chunking) [30 points]

In [30]:
kb_text = """
Queen's University at Kingston,[3][12][13] commonly known as Queen's University or simply Queen's, is a public research university in Kingston,
Ontario, Canada. Queen's holds more than 1,400 hectares (3,500 acres) of land throughout Ontario and owns Herstmonceux Castle in East Sussex,
England.[9] Queen's is organized into eight faculties and schools.

The Church of Scotland established Queen's College in October 1841 via a royal charter from Queen Victoria. The first classes, intended to prepare
students for the ministry, were held 7 March 1842, with 15 students and two professors.[14] In 1869, Queen's was the first Canadian university west
of the Maritime provinces to admit women.[3] In 1883, a women's college for medical education affiliated with Queen's University was established after
 male staff and students reacted with hostility to the admission of women to the university's medical classes.[15][16] In 1912, Queen's ended its
 affiliation with the Presbyterian Church,[12] and adopted its present name.[17][3] During the mid-20th century, the university established several
 faculties and schools and expanded its campus with the construction of new facilities.

Queen's is a co-educational university with more than 33,842 students and over 131,000 alumni living worldwide.[7][18] Notable alumni include
government officials, academics, business leaders and 62 Rhodes Scholars.[19] As of 2022, five Nobel Laureates and one Turing Award winner have
been affiliated with the university.

The university funds several magazines and journals, among which are the Queen's Quarterly that has been published since 1893.
"""

# Manual chunking — treat each paragraph as a chunk
def chunking(text):
    """
    Manually chunk a text document into smaller pieces.
    This function avoids one-liners and shows each step clearly.
    """

    # Step 1 — Split the text into raw lines
    raw_lines = text.split('\n')

    # Step 2 — Prepare an empty list for chunks
    chunks = []

    # Step 3 — Loop through each line
    for line in raw_lines:

        # Step 3a — Remove leading/trailing whitespace
        cleaned_line = line.strip()

        # Step 3b — Skip empty lines
        if len(cleaned_line) == 0:
            continue

        # Step 3c — Add cleaned line as a chunk
        chunks.append(cleaned_line)

    # Step 4 — Return the final list of chunks
    return chunks

## 📌 Section 2 — Embed Chunks Using Sentence Transformers

In [31]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Choose a small free embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Compute embeddings for each chunk
chunks = chunking(kb_text)
chunk_vectors = embedder.encode(chunks, convert_to_numpy=True)

print("Chunk vectors shape:", chunk_vectors.shape)

Chunk vectors shape: (13, 384)


## 📌 Section 3 — Manual Cosine Similarity Function [5 points]
Given two vectors **a** and **b**, the cosine similarity is defined as:

$$
\text{cosine similarity}(\mathbf{a}, \mathbf{b}) =
\frac{\mathbf{a} \cdot \mathbf{b}}
{\left\lVert \mathbf{a} \right\rVert \, \left\lVert \mathbf{b} \right\rVert}
$$

Where:

**Dot product:**

$$
\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i b_i
$$

**Magnitude (norm) of vector a:**

$$
\left\lVert \mathbf{a} \right\rVert = \sqrt{\sum_{i=1}^{n} a_i^2}
$$

**Magnitude (norm) of vector b:**

$$
\left\lVert \mathbf{b} \right\rVert = \sqrt{\sum_{i=1}^{n} b_i^2}
$$

The value ranges from:

- **1** → vectors point in the same direction  
- **0** → vectors are orthogonal  
- **−1** → vectors point in opposite directions  


In [32]:
def cosine_sim(a, b):
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    cosine_sim = dot_product / (norm_a * norm_b)
    return cosine_sim

## 📌 Section 4 — Manual Retrieval [20 points]

In [33]:
def retrieve(query, top_k=2):
    '''
    Retrieve the top_k most relevant chunks for the input query
    using manual cosine similarity
    '''

    # ---------------------------------------------------------
    # Step 1: Convert the query text into an embedding vector
    # ---------------------------------------------------------
    # embedder.encode returns a list; we take the first element.
    q_vec = embedder.encode([query], convert_to_numpy=True)[0]

    # ---------------------------------------------------------
    # Step 2: Compute cosine similarity with every chunk vector
    # ---------------------------------------------------------
    cosine_sims = []
    for vec in chunk_vectors:
        score = cosine_sim(q_vec, vec)
        cosine_sims.append(score)
    # ---------------------------------------------------------
    # Step 3: Find indices of the top_k most similar chunks
    # ---------------------------------------------------------
    top_k_idx = sorted(
        enumerate(cosine_sims),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]
    # ---------------------------------------------------------
    # Step 4: Return a list of tuples:
    # (chunk_index, similarity_score, chunk_text)
    # ---------------------------------------------------------
    results = []
    for idx, score in top_k_idx:
        results.append((idx, score, chunks[idx]))
    return results

## 📌 Section 5 — Load a Small LLM

In [34]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)


## 📌 Section 6 — Raw LLM Output without RAG

In [35]:
def raw_generate(query, top_k=2, max_new_tokens=128, return_full=False):

    # simple direct prompt
    prompt = f"""
                You are a helpful assistant. Answer the following question:

                Question: {query}
                Answer:
            """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        inputs["input_ids"],
        max_new_tokens=max_new_tokens,
        temperature=0.7
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


## 📌 Section 6 — RAG Pipeline (From Scratch) [10 points]

In [36]:
def rag_generate(query, top_k=2, max_new_tokens=200, return_full=False):

    # 1. Manual Retrieval
    # Retrieve the top_k most relevant chunks for the input query using manual cosine similarity.
    retrieved = retrieve(query, top_k=top_k)
    retrieved_text = "\n".join([f"- {txt}" for _, _, txt in retrieved])

    # 2. Construct Prompt （change this to fill in retrieved text）
    prompt = f"""
                You are a helpful assistant. Answer the following question:

                Question: {query}
                Answer:
                
                Use the following relevant information:
                {retrieved_text}
            """

    # 3. Generate
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output_ids = model.generate(
        inputs["input_ids"],
        max_new_tokens=200,
        temperature=0.7
    )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


## 📌 Section 7 — Test Retrieval [10 points]

In [37]:
query = "Where is Queen's University?"

print("=== Retrieved Chunks ===\n")
for idx, sim, txt in retrieve(query):
    print(f"[{idx}] (score={sim:.3f})\n{txt}\n")

print("\n=== RAG Answer ===\n")
## RAG answer here
rag_answer = rag_generate(query, top_k=2, max_new_tokens=200, return_full=False)
print(rag_answer)

print("======================================")
print("            RAW LLM Output")
## RAW LLM Output here
raw_answer = raw_generate(query, top_k=2, max_new_tokens=200, return_full=False)
print(raw_answer)


=== Retrieved Chunks ===

[0] (score=0.735)
Queen's University at Kingston,[3][12][13] commonly known as Queen's University or simply Queen's, is a public research university in Kingston,

[9] (score=0.718)
Queen's is a co-educational university with more than 33,842 students and over 131,000 alumni living worldwide.[7][18] Notable alumni include


=== RAG Answer ===


                You are a helpful assistant. Answer the following question:

                Question: Where is Queen's University?
                Answer:

                Use the following relevant information:
                - Queen's University at Kingston,[3][12][13] commonly known as Queen's University or simply Queen's, is a public research university in Kingston,
- Queen's is a co-educational university with more than 33,842 students and over 131,000 alumni living worldwide.[7][18] Notable alumni include
            
            - Sir Isaac Newton, the renowned mathematician and physicist, who studied at Queen's

## 📌 Section 8 — Load `vald.csv` and run evaluation

This cell loads the provided `vald.csv`, rebuilds a small knowledge base from the contexts, and reports **Exact Match (EM)** and **token-level F1**.

> Assumes you have already implemented: `chunking`, `cosine_sim`, `retrieve`, `rag_generate`, and loaded the embedder + LLM in earlier sections.


In [42]:
import json
import pandas as pd
import numpy as np
import re, string

# -----------------------------
# Load validation CSV (100 rows)
# -----------------------------
df = pd.read_csv("vald.csv")
print("Validation size:", len(df))

# -----------------------------
# Build KB chunks with mapping
# (preserve which context each chunk came from)
# -----------------------------
contexts = df["context"].drop_duplicates().reset_index(drop=True)
ctx2id = {c:i for i,c in enumerate(contexts.tolist())}

corpus_chunks = []
chunk_ctx_id = []

for cid, ctx in enumerate(contexts.tolist()):
    chs = chunking(ctx)
    corpus_chunks.extend(chs)
    chunk_ctx_id.extend([cid]*len(chs))

corpus_emb = embedder.encode(corpus_chunks, convert_to_numpy=True)

def retrieve_eval(query, top_k=5):
    q_vec = embedder.encode([query], convert_to_numpy=True)[0]
    sims = np.array([cosine_sim(q_vec, v) for v in corpus_emb])
    top_k = min(top_k, len(sims))
    top_idx = np.argsort(-sims)[:top_k]
    return [(int(i), float(sims[i]), corpus_chunks[int(i)], int(chunk_ctx_id[int(i)])) for i in top_idx]

# -----------------------------
# SQuAD-style normalization + EM/F1
# -----------------------------
def normalize_answer(s):
    s = str(s).lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = " ".join(s.split())
    return s

def exact_match(pred, gold):
    return 1.0 if normalize_answer(pred) == normalize_answer(gold) else 0.0

def f1_score(pred, gold):
    pred_toks = normalize_answer(pred).split()
    gold_toks = normalize_answer(gold).split()
    if len(pred_toks) == 0 and len(gold_toks) == 0:
        return 1.0
    if len(pred_toks) == 0 or len(gold_toks) == 0:
        return 0.0
    common = {}
    for t in pred_toks:
        common[t] = common.get(t, 0) + 1
    num_same = 0
    for t in gold_toks:
        if common.get(t, 0) > 0:
            num_same += 1
            common[t] -= 1
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_toks)
    recall = num_same / len(gold_toks)
    return 2 * precision * recall / (precision + recall)

def best_em_f1(pred, golds):
    em = max(exact_match(pred, g) for g in golds)
    f1 = max(f1_score(pred, g) for g in golds)
    return em, f1

# -----------------------------
# ROUGE-L (F1) — common project metric for span-ish QA
# -----------------------------
def lcs_len(x, y):
    # x,y are token lists
    dp = [[0]*(len(y)+1) for _ in range(len(x)+1)]
    for i in range(1, len(x)+1):
        for j in range(1, len(y)+1):
            if x[i-1] == y[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[-1][-1]

def rouge_l_f1(pred, gold):
    pt = normalize_answer(pred).split()
    gt = normalize_answer(gold).split()
    if len(pt) == 0 and len(gt) == 0:
        return 1.0
    if len(pt) == 0 or len(gt) == 0:
        return 0.0
    lcs = lcs_len(pt, gt)
    prec = lcs / len(pt)
    rec = lcs / len(gt)
    if prec + rec == 0:
        return 0.0
    return 2 * prec * rec / (prec + rec)

def best_rouge_l(pred, golds):
    return max(rouge_l_f1(pred, g) for g in golds)

# -----------------------------
# Evaluation: RAW vs RAG
# + retrieval metrics that are standard in RAG projects
# -----------------------------
TOP_K = 5

raw_em, raw_f1, raw_rl = [], [], []
rag_em, rag_f1, rag_rl = [], [], []

ctx_hit, ctx_mrr = [], []
ans_in_topk = []  # answer string appears in any retrieved chunk (proxy for "answer recall@k")

for _, row in df.iterrows():
    q = row["question"]
    golds = json.loads(row["answers_text"])
    true_cid = ctx2id[row["context"]]

    # retrieval metrics (independent of the generator)
    retrieved = retrieve_eval(q, top_k=TOP_K)
    hit = any(cid == true_cid for *_, cid in retrieved)
    ctx_hit.append(1.0 if hit else 0.0)

    mrr = 0.0
    for rank, item in enumerate(retrieved, start=1):
        cid = item[-1]
        if cid == true_cid:
            mrr = 1.0 / rank
            break
    ctx_mrr.append(mrr)

    retrieved_text_join = " ".join([txt for _,_,txt,_ in retrieved])
    ans_hit = any(normalize_answer(a) in normalize_answer(retrieved_text_join) for a in golds)
    ans_in_topk.append(1.0 if ans_hit else 0.0)

    # predictions
    pred_raw = raw_generate(q, max_new_tokens=64, return_full=False)
    pred_rag = rag_generate(q, top_k=TOP_K, max_new_tokens=64, return_full=False)

    # QA metrics
    em, f1 = best_em_f1(pred_raw, golds); raw_em.append(em); raw_f1.append(f1); raw_rl.append(best_rouge_l(pred_raw, golds))
    em, f1 = best_em_f1(pred_rag, golds); rag_em.append(em); rag_f1.append(f1); rag_rl.append(best_rouge_l(pred_rag, golds))

# summary "evaluation matrix"
retr_report = pd.DataFrame([{
    "Retriever metrics": f"top_k={TOP_K}",
    "Context Hit@k": float(np.mean(ctx_hit)),
    "Context MRR": float(np.mean(ctx_mrr)),
    "Answer Recall@k": float(np.mean(ans_in_topk)),
}])

display(retr_report)

Validation size: 100


,Retriever metrics,Context Hit@k,Context MRR,Answer Recall@k
0,top_k=5,0.99,0.955,0.99
